In [32]:
## PACKAGES
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import umap.umap_ as umap
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import timm
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import seaborn as sn
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import sys
sys.path.insert(1,'C:\Zmyfiles\codes\research\KPCA-DA\KPCA_DA\examples\Scripts')
from pyDeepInsightv3 import ImageTransformer

In [33]:
#import dataset
pfile1 = r"..\Data\cfRNA\GSE174302.csv"
pfile2 = r"..\Data\cfRNA\GSE142987.csv"

expr1 = pd.read_csv(pfile1, encoding='utf-8', engine='python')
expr2 = pd.read_csv(pfile2, encoding='utf-8', engine='python')

In [34]:
# Clean gene names so both datasets use plain base Ensembl IDs (no version, no pipe fields)
# GSE174302 format: "ENSG00000000003.14|mRNA|TSPAN6|..." → "ENSG00000000003"
expr1['feature'] = (expr1['feature']
                    .str.split('|').str[0]   # take first pipe-field (Ensembl ID.version)
                    .str.split('.').str[0])  # strip .version suffix

# GSE142987 format: "ENSG00000243485.5" → "ENSG00000243485"
expr2['Sample_name'] = expr2['Sample_name'].str.split('.').str[0]

# Drop any duplicate gene IDs (keep first occurrence)
expr1 = expr1.drop_duplicates(subset='feature', keep='first')
expr2 = expr2.drop_duplicates(subset='Sample_name', keep='first')

print(f"expr1 genes: {len(expr1)}  sample names: {expr1['feature'].iloc[0]!r}")
print(f"expr2 genes: {len(expr2)}  sample names: {expr2['Sample_name'].iloc[0]!r}")

expr1 genes: 139311  sample names: 'ENSG00000000003'
expr2 genes: 84634  sample names: 'ENSG00000243485'


In [35]:
# Set gene name as row index and remove all-zero genes
expr1 = expr1.set_index('feature')       # genes x samples
expr2 = expr2.set_index('Sample_name')   # genes x samples

expr1 = expr1.loc[expr1.sum(axis=1) > 0]
expr2 = expr2.loc[expr2.sum(axis=1) > 0]

print(f"expr1: {expr1.shape}  expr2: {expr2.shape}")

expr1: (130884, 230)  expr2: (83976, 65)


In [36]:
def vst_normalize(counts_df):
    """
    Variance Stabilizing Transformation for RNA-seq count data.
    DESeq2-style: size-factor normalization (median-of-ratios) followed by
    the analytical VST for a negative binomial dispersion model.
    Input:  genes x samples DataFrame of raw integer counts
    Output: genes x samples DataFrame of VST-stabilized float values
    """
    counts = counts_df.values.astype(float)

    # Size factors via DESeq2 median-of-ratios method
    log_counts = np.log(counts + 1)
    geo_log_mean = log_counts.mean(axis=1)           # per-gene geometric mean (log space)
    valid = geo_log_mean > 0
    log_ratios = log_counts[valid] - geo_log_mean[valid, np.newaxis]
    size_factors = np.exp(np.median(log_ratios, axis=0))   # one per sample

    # Normalize by size factors
    norm = counts / size_factors[np.newaxis, :]

    # Per-gene dispersion via method-of-moments on negative binomial:
    # Var = mu + alpha * mu^2  =>  alpha = (Var - mu) / mu^2
    mu    = norm.mean(axis=1, keepdims=True)
    var   = norm.var(axis=1,  keepdims=True)
    alpha = np.where(mu > 0, (var - mu) / (mu ** 2 + 1e-8), 1.0)
    alpha = np.clip(alpha, 1e-4, 1e4)

    # Analytical VST:  2/sqrt(alpha) * arcsinh( sqrt(alpha * x) )
    vst = (2.0 / np.sqrt(alpha)) * np.arcsinh(np.sqrt(alpha * norm))

    return pd.DataFrame(vst, index=counts_df.index, columns=counts_df.columns)

vst1 = vst_normalize(expr1)
vst2 = vst_normalize(expr2)

print("VST1:", vst1.shape, "  range [{:.2f}, {:.2f}]".format(vst1.values.min(), vst1.values.max()))
print("VST2:", vst2.shape, "  range [{:.2f}, {:.2f}]".format(vst2.values.min(), vst2.values.max()))

VST1: (130884, 230)   range [0.00, 19.49]
VST2: (83976, 65)   range [0.00, 24.26]


In [37]:
# Transpose to samples x genes for ML
X1 = vst1.T.values.astype(np.float32)   # source: GSE174302
X2 = vst2.T.values.astype(np.float32)   # target: GSE142987

# Binary labels: Cancer vs Normal
def label_gse174302(name):
    return 'Normal' if name.split('-')[0].startswith('NC') else 'Cancer'

def label_gse142987(name):
    return 'Normal' if name.startswith('H') else 'Cancer'

y1_raw = np.array([label_gse174302(s) for s in vst1.columns])
y2_raw = np.array([label_gse142987(s) for s in vst2.columns])

le = LabelEncoder()
le.fit(['Cancer', 'Normal'])
Y1 = le.transform(y1_raw)
Y2 = le.transform(y2_raw)

print("X1 (source):", X1.shape, " labels:", dict(zip(*np.unique(y1_raw, return_counts=True))))
print("X2 (target):", X2.shape, " labels:", dict(zip(*np.unique(y2_raw, return_counts=True))))
print("Encoded classes:", list(le.classes_))

X1 (source): (230, 130884)  labels: {'Cancer': 184, 'Normal': 46}
X2 (target): (65, 83976)  labels: {'Cancer': 35, 'Normal': 30}
Encoded classes: ['Cancer', 'Normal']


In [38]:
# Select top 70% most variant genes independently per dataset
def top70_var(X, gene_names):
    var = X.var(axis=0)
    mask = var >= np.percentile(var, 50)   # 50th percentile cutoff = top 70%
    return X[:, mask], np.array(gene_names)[mask]

X1, genes1 = top70_var(X1, vst1.index.tolist())
X2, genes2 = top70_var(X2, vst2.index.tolist())

print(f"X1: {X1.shape}  genes retained: {len(genes1)}")
print(f"X2: {X2.shape}  genes retained: {len(genes2)}")

X1: (230, 65442)  genes retained: 65442
X2: (65, 41988)  genes retained: 41988


In [39]:
# MinMaxScale each dataset independently (different gene sets)
mms1 = MinMaxScaler()
X1_norm = mms1.fit_transform(X1)

mms2 = MinMaxScaler()
X2_norm = mms2.fit_transform(X2)

print("X1_norm:", X1_norm.shape)
print("X2_norm:", X2_norm.shape)

X1_norm: (230, 65442)
X2_norm: (65, 41988)


In [40]:
# Augment target (X2) with Gaussian noise — both classes — before splitting
def augment_gaussian(X, y, target_per_class, noise_std=0.02, random_state=42):
    """Oversample each class to target_per_class by adding Gaussian noise."""
    rng = np.random.default_rng(random_state)
    X_parts, y_parts = [X], [y]
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        n_needed = target_per_class - len(idx)
        if n_needed > 0:
            chosen = rng.choice(idx, size=n_needed, replace=True)
            noise = rng.normal(0, noise_std, X[chosen].shape).astype(np.float32)
            X_parts.append(np.clip(X[chosen] + noise, 0.0, 1.0))
            y_parts.append(np.full(n_needed, cls))
    return np.vstack(X_parts), np.concatenate(y_parts)

classes, counts = np.unique(Y2, return_counts=True)
target_n = int(counts.max() * 2)   # grow each class to 2× the largest class
print("Before augmentation:", dict(zip(le.classes_[classes], counts)))

X2_aug, Y2_aug = augment_gaussian(X2_norm, Y2, target_per_class=target_n)
_, aug_counts = np.unique(Y2_aug, return_counts=True)
print("After  augmentation:", dict(zip(le.classes_, aug_counts)))
print("X2_aug:", X2_aug.shape)

Before augmentation: {'Cancer': 35, 'Normal': 30}
After  augmentation: {'Cancer': 70, 'Normal': 70}
X2_aug: (140, 41988)


In [41]:
# Split augmented target: 70% adapt (reserved for KPCA-DA later), 30% test
X2_adapt, X2_test, Y2_adapt, Y2_test = train_test_split(
    X2_aug, Y2_aug, test_size=0.3, random_state=42, stratify=Y2_aug)

print("X2_adapt:", X2_adapt.shape, " labels:", dict(zip(*np.unique(Y2_adapt, return_counts=True))))
print("X2_test: ", X2_test.shape,  " labels:", dict(zip(*np.unique(Y2_test,  return_counts=True))))

X2_adapt: (98, 41988)  labels: {0: 49, 1: 49}
X2_test:  (42, 41988)  labels: {0: 21, 1: 21}


In [ ]:
# DeepInsight image transformation
from sklearn.decomposition import KernelPCA

pixel_size = (227, 227)

it1 = ImageTransformer(
    feature_extractor=KernelPCA(n_components=2, kernel='rbf', gamma=0.2),
    pixels=pixel_size)

it2 = ImageTransformer(
    feature_extractor=KernelPCA(n_components=2, kernel='rbf', gamma=0.2),
    pixels=pixel_size)

print("Fitting ImageTransformer on source (X1_norm)...")
X_train_img = it1.fit_transform(X1_norm)

print("Fitting ImageTransformer on target adapt (X2_adapt)...")
X_adapt_img = it2.fit_transform(X2_adapt)

print("Transforming target test with same transformer...")
X_test_img = it2.transform(X2_test)

print("Source images :", X_train_img.shape)
print("Adapt images  :", X_adapt_img.shape)
print("Test images   :", X_test_img.shape)

In [ ]:
# ResNet-50 — model, weighted loss, optimiser, data loaders
num_classes = len(le.classes_)
le_mapping  = dict(zip(range(num_classes), le.classes_))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = timm.create_model('resnet50d', pretrained=True, num_classes=num_classes).to(device)

batch_size = 16
preprocess = transforms.Compose([transforms.ToTensor()])

# Inverse-frequency class weights on source labels
class_counts  = np.bincount(Y1, minlength=num_classes)
weights       = (1.0 / class_counts) * num_classes / 2.0
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
optimizer     = optim.Adam(net.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler     = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Train loader — source images only
X_train_tensor = torch.stack([preprocess(img) for img in X_train_img]).float().to(device)
y_train_tensor = torch.from_numpy(Y1).long().to(device)
trainloader    = DataLoader(TensorDataset(X_train_tensor, y_train_tensor),
                            batch_size=batch_size, shuffle=True)

# Test loader — target test images
X_test_tensor = torch.stack([preprocess(img) for img in X_test_img]).float().to(device)
y_test_tensor = torch.from_numpy(Y2_test).long().to(device)
testloader    = DataLoader(TensorDataset(X_test_tensor, y_test_tensor),
                           batch_size=batch_size, shuffle=False)

print(f"Train: {len(X_train_tensor)} source images | Test: {len(X_test_tensor)} target images | Device: {device}")

In [ ]:
# Train ResNet-50 on source only
num_epochs   = 30
train_losses = []

net.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in trainloader:
        optimizer.zero_grad()
        loss = criterion(net(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    epoch_loss = running_loss / len(trainloader)
    train_losses.append(epoch_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1:2d}/{num_epochs}]  Loss: {epoch_loss:.4f}")

print("Training complete.")

In [ ]:
# Evaluate on target test set (no adaptation)
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

net.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for inputs, labels in testloader:
        preds = net(inputs).argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

print(f"Accuracy:          {accuracy_score(all_true, all_preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(all_true, all_preds):.4f}")
print(f"F1 (weighted):     {f1_score(all_true, all_preds, average='weighted'):.4f}")

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(4, 3))
sn.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Target Test — No Adaptation (Baseline)')
plt.tight_layout()
plt.show()

### With Domain Adaptation

In [42]:
from kpcada import KPCA_DA, tune_lambda
from pyDeepInsightv3.image_transformer_new import DomainAdaptiveImageTransformer

In [43]:
# Pre-align source and target to common genes and reduce to top 5000 most variable.
# Doing this once avoids repeated expensive gene-alignment inside tune_lambda/fit_transform.
genes1_idx = {g: i for i, g in enumerate(genes1)}
genes2_idx = {g: i for i, g in enumerate(genes2)}

common_kpca = np.array(sorted(set(genes1).intersection(set(genes2))))
print(f"Common genes: {len(common_kpca)}")

idx1 = np.array([genes1_idx[g] for g in common_kpca])
idx2 = np.array([genes2_idx[g] for g in common_kpca])

X1_common       = X1_norm[:, idx1]
X2_adapt_common = X2_adapt[:, idx2]
X2_test_common  = X2_test[:, idx2]

# Keep top 5000 by source variance — enough signal for a 2-component embedding
top5k         = np.argsort(X1_common.var(axis=0))[-5000:]
X1_kpca       = X1_common[:, top5k]
X2_adapt_kpca = X2_adapt_common[:, top5k]
X2_test_kpca  = X2_test_common[:, top5k]
genes_kpca    = common_kpca[top5k]

print(f"KPCA-DA input — source: {X1_kpca.shape}  adapt: {X2_adapt_kpca.shape}  test: {X2_test_kpca.shape}")

Common genes: 39671
KPCA-DA input — source: (230, 5000)  adapt: (98, 5000)  test: (42, 5000)


In [ ]:
# Grid search for optimal lambda — genes already aligned so no internal alignment overhead
print("Running lambda grid search...")

summary = tune_lambda(
    X_source=X1_kpca.T,          # genes x samples (pre-aligned, 5000 genes)
    X_target=X2_adapt_kpca.T,
    genes_source=genes_kpca,
    genes_target=genes_kpca,      # identical — alignment is a no-op
    lambda_values=[0.001, 0.01, 0.1],   # 3 values instead of 5
    n_components=2,
    kernel="rbf",
    auto_gamma=True,
    lr=0.001,
    epochs=100,                   # 100 instead of 500
    verbose=False,
)

best_lambda = summary["best_lambda_mmd"]
print(summary["results_df"][["lambda", "final_mmd", "final_loss"]].to_string(index=False))
print(f"\nBest lambda (lowest MMD): {best_lambda}")

Running lambda grid search...
 lambda  final_mmd  final_loss
  0.001   0.430985    0.651106
  0.010   0.431108    2.624487
  0.100   0.432197   21.597881

Best lambda (lowest MMD): 0.001


In [ ]:
# Fit DomainAdaptiveImageTransformer with best lambda on pre-aligned 5000-gene matrices
da_transformer = DomainAdaptiveImageTransformer(
    kpca_da_params={
        "n_components": 2,
        "kernel":       "rbf",
        "auto_gamma":   True,
        "lambda_reg":   best_lambda,
        "lr":           0.001,
        "epochs":       200,          # 200 instead of 500
        "verbose":      False,
    },
    pixels=(227, 227)
)

# Z_source, Z_adapt = da_transformer.fit_transform(
#     X_source=X1_kpca,
#     X_target=X2_adapt_kpca,
#     genes_source=genes_kpca,
#     genes_target=genes_kpca,          # pre-aligned — no internal alignment cost
#     plot=True
# )

Z_source, Z_adapt = da_transformer.fit_transform(
    X_source=X1_norm,
    X_target=X2_adapt,
    genes_source=genes1,
    genes_target=genes2,          # pre-aligned — no internal alignment cost
    plot=True
)

source_images_da = da_transformer.transform_source(X1_norm)
adapt_images_da  = da_transformer.transform_target(X2_adapt)
test_images_da   = da_transformer.transform_target(X2_test)

print("Source images :", source_images_da.shape)
print("Adapt images  :", adapt_images_da.shape)
print("Test images   :", test_images_da.shape)

Applying KPCA-DA for domain adaptation...


c:\Users\sheli\anaconda3\envs\SheliaWorkspace\lib\site-packages\sklearn\metrics\pairwise.py:611: RuntimeWarning: overflow encountered in scalar multiply
  + (x_density * n_samples_X * y_density * n_samples_Y)


In [ ]:
# ResNet-50 — fresh model, train on source + adapt combined
device_da = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net_da = timm.create_model('resnet50d', pretrained=True, num_classes=num_classes).to(device_da)

preprocess_da = transforms.Compose([transforms.ToTensor()])
batch_size_da = 16

# Combine source and adapt for training
train_images_all = np.concatenate([source_images_da, adapt_images_da], axis=0)
train_labels_all = np.concatenate([Y1, Y2_adapt])

# Inverse-frequency class weights on combined training set
cc = np.bincount(train_labels_all, minlength=num_classes)
cw = torch.tensor((1.0 / cc) * num_classes / 2.0, dtype=torch.float32).to(device_da)
criterion_da = nn.CrossEntropyLoss(weight=cw)
optimizer_da = optim.Adam(net_da.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_da = torch.optim.lr_scheduler.StepLR(optimizer_da, step_size=10, gamma=0.5)

X_train_da = torch.stack([preprocess_da(img) for img in train_images_all]).float().to(device_da)
y_train_da = torch.tensor(train_labels_all, dtype=torch.long).to(device_da)
trainloader_da = DataLoader(TensorDataset(X_train_da, y_train_da),
                            batch_size=batch_size_da, shuffle=True)

X_test_da = torch.stack([preprocess_da(img) for img in test_images_da]).float().to(device_da)
y_test_da = torch.tensor(Y2_test, dtype=torch.long).to(device_da)
testloader_da = DataLoader(TensorDataset(X_test_da, y_test_da),
                           batch_size=batch_size_da, shuffle=False)

print(f"Train: {len(X_train_da)} images (source={len(source_images_da)}, adapt={len(adapt_images_da)})")
print(f"Test : {len(X_test_da)} images | Device: {device_da}")

In [ ]:
# Train ResNet-50 on source + adapt
num_epochs_da = 30

net_da.train()
for epoch in range(num_epochs_da):
    running_loss = 0.0
    for inputs, labels in trainloader_da:
        optimizer_da.zero_grad()
        loss = criterion_da(net_da(inputs), labels)
        loss.backward()
        optimizer_da.step()
        running_loss += loss.item()
    scheduler_da.step()
    epoch_loss = running_loss / len(trainloader_da)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1:2d}/{num_epochs_da}]  Loss: {epoch_loss:.4f}")

print("Training complete.")

In [ ]:
# Evaluate on target test set (with KPCA-DA adaptation)
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

net_da.eval()
da_preds, da_true = [], []
with torch.no_grad():
    for inputs, labels in testloader_da:
        preds = net_da(inputs).argmax(dim=1)
        da_preds.extend(preds.cpu().numpy())
        da_true.extend(labels.cpu().numpy())

da_preds = np.array(da_preds)
da_true  = np.array(da_true)

print(f"Accuracy:          {accuracy_score(da_true, da_preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(da_true, da_preds):.4f}")
print(f"F1 (weighted):     {f1_score(da_true, da_preds, average='weighted'):.4f}")

cm_da = confusion_matrix(da_true, da_preds)
plt.figure(figsize=(4, 3))
sn.heatmap(cm_da, annot=True, fmt='d', cmap='Greens',
           xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Target Test — With KPCA-DA Adaptation')
plt.tight_layout()
plt.show()